In [4]:
import numpy as np
from PIL import Image
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

In [3]:
def block_averages(arr, n_rows, n_cols):
    bh = arr.shape[0] // n_rows
    bw = arr.shape[1] // n_cols
    avg = np.zeros((n_rows, n_cols))
    for i in range(n_rows):
        for j in range(n_cols):
            block = arr[i*bh:(i+1)*bh, j*bw:(j+1)*bw]
            avg[i, j] = block.mean()
    return avg

In [19]:
# Setup and resizing

numberOfTilesInHeight = 15
numberOfLabyrinthsToFind = 2
sizeOfLineInLaby = 15
imageFile = "lincoln.png"

# Load image as grayscale
img = Image.open(imageFile).convert("L")
targetImage = np.array(img)

#Find the dimensions of the image
heightTarget, widthTarget = targetImage.shape

numberOfTilesInWidth = int(np.floor(widthTarget / heightTarget * (numberOfTilesInHeight + 1)))

#New sizes of image
heightTarget = int(np.floor(heightTarget / numberOfTilesInHeight) * numberOfTilesInHeight)
widthTarget  = int(np.floor(widthTarget / numberOfTilesInWidth) * numberOfTilesInWidth)

#resize the image

targetImage = Image.fromarray(targetImage).resize((widthTarget, heightTarget))

#print(targetImage.size)
#display(targetImage)

#convert back to np array
targetImage = np.array(targetImage)

pixelsInHeight = int(np.floor(heightTarget/numberOfTilesInHeight))
pixelsInWidth  = int(np.floor(widthTarget/numberOfTilesInWidth))

#print(pixelsInHeight, pixelsInWidth)

In [23]:
#Find pixel averages over subblocks

pixelAverage = np.zeros((numberOfTilesInHeight, numberOfTilesInWidth))

pixelRow = 0
for i in range(numberOfTilesInHeight):
    pixelCol = 0
    for j in range(numberOfTilesInWidth):
        block = targetImage[
            pixelRow:pixelRow + pixelsInHeight,
            pixelCol:pixelCol + pixelsInWidth
        ]
        pixelAverage[i, j] = block.mean()
        pixelCol += pixelsInWidth
    pixelRow += pixelsInHeight

In [32]:
#Create indexing of decision variables
# x_i,j,k is whether piece k is placed in block (i,j) where

numberOfTotalTiles = 7
numberOfDecisionVariables = 0
gridIndexing = np.zeros(
    (numberOfTilesInHeight,numberOfTilesInWidth,numberOfTotalTiles), dtype=int)

indexToGrid = []

for i in range(numberOfTilesInHeight):
    for j in range(numberOfTilesInWidth):
        for k in range(numberOfTotalTiles):
            numberOfDecisionVariables += 1
            gridIndexing[i,j,k] = numberOfDecisionVariables
            indexToGrid.append([i, j, k])

indexToGrid = np.array(indexToGrid)

In [33]:
#Create objective function

objectiveFunction = np.zeros(numberOfDecisionVariables)

for i in range(numberOfTilesInHeight):
    for j in range(numberOfTilesInWidth):
        for k in range(numberOfTotalTiles-1):
            index = gridIndexing[i,j,k] - 1
            objectiveFunction[index] = np.square(pixelAverage[i,j])
        
        index = gridIndexing[i,j,numberOfTotalTiles-1] - 1
        objectiveFunction[index] = np.square((255 - pixelAverage[i,j]))  

In [34]:
#create constraints - one row per constraint